In [1]:
# Execute for google colab only
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Convert Custom Recorded Noise File to Desired Format

In [2]:
import librosa, soundfile as sf, numpy as np, pathlib, random, glob, os, tqdm
from IPython.display import Audio, display

BASE_PATH = "/content/drive/MyDrive/colab-data/audio-denoising"

CLEAN_AUDIO_PATH = os.path.join(BASE_PATH, "LibriSpeech", "test-clean")

REAL_NOISE_PATH = os.path.join(BASE_PATH, "data", "real-noise-recordings")

In [3]:
def convert_audio_for_noise_training(
    input_path,
    output_path,
    target_sr=16000,
    start_sec: float | None = None, # New: Start time in seconds
    end_sec: float | None = None,    # New: End time in seconds,
    notebook_display: bool = False
):
    """
    Loads an audio file, converts it to mono, resamples to target_sr.
    If start_sec and/or end_sec are provided, extracts the specified
    segment. Saves the result as a FLAC file.

    Args:
        input_path (str): Path to the input audio file (e.g., MP3, WAV).
        output_path (str): Path to save the converted FLAC file.
        target_sr (int): The target sample rate (default: 16000).
        start_sec (float | None): The start time of the segment to extract
                                  in seconds. If None, starts from the beginning.
        end_sec (float | None): The end time of the segment to extract
                                in seconds. If None, goes to the end.
    """
    try:
        # Load the audio file
        # - sr=target_sr automatically resamples the audio to 16kHz upon loading
        # - mono=True automatically converts the audio to mono upon loading
        print(f"Loading and converting '{os.path.basename(input_path)}'...")
        audio_data, sr = librosa.load(input_path, sr=target_sr, mono=True)
        original_duration_sec = len(audio_data) / sr
        print(f"Successfully loaded. Sample rate: {sr}, Original duration: {original_duration_sec:.2f}s")

        # --- New section: Extract segment using start/end times ---
        start_sample = 0
        end_sample = len(audio_data) # Default to full length

        valid_slice = True
        if start_sec is not None:
            if start_sec < 0:
                print(f"Warning: start_sec ({start_sec}) is negative. Using 0.")
                start_sec = 0
            start_sample = int(start_sec * target_sr)
            if start_sample >= len(audio_data):
                 print(f"Warning: start_sec ({start_sec}s) is beyond audio duration ({original_duration_sec:.2f}s). Cannot extract segment.")
                 valid_slice = False

        if end_sec is not None and valid_slice:
            if end_sec < 0:
                 print(f"Warning: end_sec ({end_sec}) is negative. Cannot use negative end time.")
                 valid_slice = False
            else:
                end_sample = int(end_sec * target_sr)
                if end_sample > len(audio_data):
                    print(f"Warning: end_sec ({end_sec}s) is beyond audio duration ({original_duration_sec:.2f}s). Clipping to end.")
                    end_sample = len(audio_data) # Clip to actual end if specified end is too long

        # Final check: ensure start is before end
        if valid_slice and start_sample >= end_sample:
            print(f"Warning: start_sample ({start_sample}) is not before end_sample ({end_sample}). Cannot extract segment.")
            valid_slice = False

        # Perform slicing if requested and valid
        if valid_slice and (start_sec is not None or end_sec is not None):
            print(f"Extracting segment from {start_sample/sr:.2f}s to {end_sample/sr:.2f}s...")
            audio_data = audio_data[start_sample:end_sample]
            print(f"Segment extracted.")
        elif not valid_slice:
             print("Proceeding with full audio due to invalid start/end times.")
        else:
            print("No start/end times provided, using full audio.")
        # --- End of new section ---


        # Ensure output directory exists
        output_dir = os.path.dirname(output_path)
        if output_dir and not os.path.exists(output_dir):
            os.makedirs(output_dir)

        # Save the potentially sliced audio as FLAC
        print(f"Saving audio to '{os.path.basename(output_path)}'...")
        sf.write(output_path, audio_data, target_sr, format='FLAC')

        print(f"Processing complete. Output saved to: {output_path}")
        print(f"Final duration: {len(audio_data)/sr:.2f}s")

        if notebook_display:
            print("Playing original audio...")
            display(Audio(input_path))

            print("Playing converted audio...")
            display(Audio(audio_data, rate=target_sr))

    except Exception as e:
        print(f"Error converting file {input_path}: {e}")

In [ ]:
input_path = "/content/road-7016.mp3"
output_path = os.path.join(REAL_NOISE_PATH, "road-noise-1.flac")
convert_audio_for_noise_training(input_path, output_path, start_sec=0, end_sec=30, notebook_display=True)

Loading and converting 'road-7016.mp3'...
Successfully loaded. Sample rate: 16000, Original duration: 108.38s
Extracting segment from 0.00s to 30.00s...
Segment extracted.
Saving audio to 'road-noise-1.flac'...
Processing complete. Output saved to: /content/drive/MyDrive/colab-data/audio-denoising/data/real-noise-recordings/road-noise-1.flac
Final duration: 30.00s
Playing original audio...


Playing converted audio...


In [ ]:
input_path = "/content/road-7016.mp3"
# output_path = "/content/road-7016.flac"
output_path = os.path.join(REAL_NOISE_PATH, "road-noise-2.flac")
convert_audio_for_noise_training(input_path, output_path, start_sec=85, end_sec=100, notebook_display=True)

Loading and converting 'road-7016.mp3'...
Successfully loaded. Sample rate: 16000, Original duration: 108.38s
Extracting segment from 85.00s to 100.00s...
Segment extracted.
Saving audio to 'road-noise-2.flac'...
Processing complete. Output saved to: /content/drive/MyDrive/colab-data/audio-denoising/data/real-noise-recordings/road-noise-2.flac
Final duration: 15.00s
Playing original audio...


Playing converted audio...


## 2 Build a *RealMix* dataset

In [ ]:
import os, glob, random, tqdm, librosa, soundfile as sf, numpy as np

BASE = "/content/drive/MyDrive/colab-data/audio-denoising"
CLEAN_SRC = os.path.join(BASE, "LibriSpeech", "test-clean")
NOISE_SRC = os.path.join(BASE, "data", "real-noise-recordings")
OUT_CLEAN = os.path.join(BASE, "data", "realmixed", "clean")
OUT_NOISY = os.path.join(BASE, "data", "realmixed", "noisy")
os.makedirs(OUT_CLEAN, exist_ok=True)
os.makedirs(OUT_NOISY, exist_ok=True)

# cache resampled noise
cache = {}
def noise16(path):
    if path not in cache:
        y, sr = librosa.load(path, sr=None, mono=True)
        if sr != 16000:
            y = librosa.resample(y, sr, 16000)
        cache[path] = y.astype(np.float32)
    return cache[path]

clean_files = glob.glob(f"{CLEAN_SRC}/**/*.flac", recursive=True)
noise_files = glob.glob(f"{NOISE_SRC}/*")
print(f"Found {len(clean_files)} clean files and {len(noise_files)} noise files")

for cpath in tqdm.tqdm(clean_files, desc="RealMix"):
    fname = os.path.basename(cpath)
    print(fname)
    clean_out_path = os.path.join(OUT_CLEAN, fname)
    noisy_out_path = os.path.join(OUT_NOISY, fname)

    # Skip if already processed
    if os.path.exists(clean_out_path) and os.path.exists(noisy_out_path):
        continue

    y_clean, _ = librosa.load(cpath, sr=16000, mono=True)

    y_noise = noise16(random.choice(noise_files))
    if len(y_noise) < len(y_clean):
        reps = int(np.ceil(len(y_clean)/len(y_noise)))
        y_noise = np.tile(y_noise, reps)[:len(y_clean)]
    else:
        y_noise = y_noise[:len(y_clean)]

    snr = random.uniform(0, 15)                     # dB
    k = np.sqrt(np.mean(y_clean**2) /
                (np.mean(y_noise**2) * 10**(snr/10)))
    y_mixed = y_clean + k*y_noise

    sf.write(clean_out_path, y_clean, 16000,
             format="FLAC", subtype="PCM_16")
    sf.write(noisy_out_path, y_mixed, 16000,
             format="FLAC", subtype="PCM_16")

print("✔ RealMix pairs saved with identical filenames (existing files skipped)")

## 3. Create two sec segments of new data

In [5]:
import os

# ─── paths ──────────────────────────────────────────────────────────
BASE_PATH      = "/content/drive/MyDrive/colab-data/audio-denoising"
REALMIX_CLEAN  = os.path.join(BASE_PATH, "data", "realmixed", "clean")
REALMIX_NOISY  = os.path.join(BASE_PATH, "data", "realmixed", "noisy")

SEG_CLEAN_DIR  = os.path.join(BASE_PATH, "data", "two-sec-realmix", "clean")
SEG_NOISY_DIR  = os.path.join(BASE_PATH, "data", "two-sec-realmix", "noisy")

os.makedirs(SEG_CLEAN_DIR, exist_ok=True)
os.makedirs(SEG_NOISY_DIR, exist_ok=True)

In [18]:
import os
import librosa
import soundfile as sf
from tqdm import tqdm

def gather_flac_map(folder):
    """
    Recursively collect all .flac paths under 'folder',
    store them in a dict keyed by the file’s basename.
    """
    flac_map = {}
    for root, dirs, files in os.walk(folder):
        for fname in files:
            if fname.lower().endswith(".flac"):
                full_path = os.path.join(root, fname)
                flac_map[fname] = full_path
    return flac_map


def create_2sec_segments(clean_dir, noisy_dir, out_clean_dir, out_noisy_dir, sr=16000):
    """
    1) Match clean & noisy .flac by basename
    2) For each pair, split both waveforms into multiple 2-second segments
       (i.e. 2*sr = 32000 samples).
    3) Skip leftover <2s at the end of each file.
    4) Write each pair of segments with a suffix "_seg0.flac", etc.
    """
    seg_len = 2 * sr  # 2 seconds of audio at 16k

    # 1) Gather maps
    clean_map = gather_flac_map(clean_dir)
    noisy_map = gather_flac_map(noisy_dir)

    # 2) Intersection by basename
    common_names = sorted(set(clean_map.keys()).intersection(set(noisy_map.keys())))
    print(f"Found {len(common_names)} matched .flac basenames.")

    # 3) For each common file, produce segments
    for basename in tqdm(common_names, desc="Splitting into 2s segments"):
        clean_path = clean_map[basename]
        noisy_path = noisy_map[basename]

        clean_audio, _ = librosa.load(clean_path, sr=sr)
        noisy_audio, _ = librosa.load(noisy_path, sr=sr)

        # We'll only produce segments up to the min length
        L = min(len(clean_audio), len(noisy_audio))
        num_segments = L // seg_len  # skip leftover

        for i in range(num_segments):
            start = i * seg_len
            end = start + seg_len

            seg_clean = clean_audio[start:end]
            seg_noisy = noisy_audio[start:end]

            # Example filename: "myfile.flac" -> "myfile_seg0.flac"
            out_clean_name = f"{basename.replace('.flac','')}_seg{i}.flac"
            out_noisy_name = f"{basename.replace('.flac','')}_seg{i}.flac"

            sf.write(os.path.join(out_clean_dir, out_clean_name), seg_clean, sr, format="FLAC")
            sf.write(os.path.join(out_noisy_dir, out_noisy_name), seg_noisy, sr,  format="FLAC")


In [ ]:
create_2sec_segments(
    clean_dir   = REALMIX_CLEAN,
    noisy_dir   = REALMIX_NOISY,
    out_clean_dir = SEG_CLEAN_DIR,
    out_noisy_dir = SEG_NOISY_DIR,
    sr = 16000          # 2 s → 32 000 samples
)

print("✓ RealMix 2‑second dataset ready:",
      len(os.listdir(SEG_CLEAN_DIR)), "segments")

Found 2620 matched .flac basenames.


Splitting into 2s segments: 100%|██████████| 2620/2620 [24:29<00:00,  1.78it/s]


✓ RealMix 2‑second dataset ready: 8455 segments


# 4. Create New Dataset for Final Test

In [7]:
import os, glob, random, tqdm, librosa, soundfile as sf, numpy as np

BASE = "/content/drive/MyDrive/colab-data/audio-denoising"
CLEAN_SRC = os.path.join(BASE, "LibriSpeech", "dev-clean") #we used test-clean for training
NOISE_SRC = os.path.join(BASE, "data", "real-noise-recordings")  #we will use previous noise mix
OUT_CLEAN = os.path.join(BASE, "data", "realmixed-test", "clean")
OUT_NOISY = os.path.join(BASE, "data", "realmixed-test", "noisy")
os.makedirs(OUT_CLEAN, exist_ok=True)
os.makedirs(OUT_NOISY, exist_ok=True)

### 4.1 Mix Noise Samples With Clean Audio

In [11]:
MAX_FILES_TO_PROCESS = 200  # Set the maximum number of files to process


cache = {}
def noise16(path):
    if path not in cache:
        y, sr = librosa.load(path, sr=None, mono=True)
        if sr != 16000:
            y = librosa.resample(y, sr, 16000)
        cache[path] = y.astype(np.float32)
    return cache[path]

clean_files = glob.glob(f"{CLEAN_SRC}/**/*.flac", recursive=True)
noise_files = glob.glob(f"{NOISE_SRC}/*")
print(f"Found {len(clean_files)} clean files and {len(noise_files)} noise files")


processed_count = 0

for cpath in tqdm.tqdm(clean_files, desc="RealMix Test", total=min(len(clean_files), MAX_FILES_TO_PROCESS)):
    if processed_count >= MAX_FILES_TO_PROCESS:
        break


    fname = os.path.basename(cpath)
    clean_out_path = os.path.join(OUT_CLEAN, fname)
    noisy_out_path = os.path.join(OUT_NOISY, fname)

    # Skip if already processed
    if os.path.exists(clean_out_path) and os.path.exists(noisy_out_path):
        continue

    y_clean, _ = librosa.load(cpath, sr=16000, mono=True)

    y_noise = noise16(random.choice(noise_files))
    if len(y_noise) < len(y_clean):
        reps = int(np.ceil(len(y_clean)/len(y_noise)))
        y_noise = np.tile(y_noise, reps)[:len(y_clean)]
    else:
        y_noise = y_noise[:len(y_clean)]

    snr = random.uniform(0, 15)                     # dB
    k = np.sqrt(np.mean(y_clean**2) /
                (np.mean(y_noise**2) * 10**(snr/10)))
    y_mixed = y_clean + k*y_noise

    sf.write(clean_out_path, y_clean, 16000,
             format="FLAC", subtype="PCM_16")
    sf.write(noisy_out_path, y_mixed, 16000,
             format="FLAC", subtype="PCM_16")

    processed_count += 1


print("✔ RealMix pairs saved with identical filenames (existing files skipped)")

Found 2703 clean files and 4 noise files


RealMix Test: 237it [01:14,  3.17it/s]

✔ RealMix pairs saved with identical filenames (existing files skipped)


In [13]:
out_clean_files = glob.glob(f"{OUT_CLEAN}/**/*.flac", recursive=True)
out_noisy_files = glob.glob(f"{OUT_NOISY}/*")
print(f"Found {len(out_clean_files)} clean files and {len(out_noisy_files)} noise files")

Found 237 clean files and 237 noise files


In [15]:
#Sample of newly created data
print("Clean")
display(Audio(out_clean_files[0]))

print("Noisy")
display(Audio(out_noisy_files[0]))

Clean


Noisy


### 4.2 Create Two Seconds Segment

In [20]:
import os

# ─── paths ──────────────────────────────────────────────────────────
BASE_PATH      = "/content/drive/MyDrive/colab-data/audio-denoising"
TEST_REALMIX_CLEAN  = os.path.join(BASE_PATH, "data", "realmixed-test", "clean")
TEST_REALMIX_NOISY  = os.path.join(BASE_PATH, "data", "realmixed-test", "noisy")

TEST_SEG_CLEAN_DIR  = os.path.join(BASE_PATH, "data", "two-sec-realmix-test", "clean")
TEST_SEG_NOISY_DIR  = os.path.join(BASE_PATH, "data", "two-sec-realmix-test", "noisy")

os.makedirs(TEST_SEG_CLEAN_DIR, exist_ok=True)
os.makedirs(TEST_SEG_NOISY_DIR, exist_ok=True)

In [21]:
create_2sec_segments(
    clean_dir   = TEST_REALMIX_CLEAN,
    noisy_dir   = TEST_REALMIX_NOISY,
    out_clean_dir = TEST_SEG_CLEAN_DIR,
    out_noisy_dir = TEST_SEG_NOISY_DIR,
    sr = 16000          # 2 s → 32 000 samples
)

print("✓ RealMix 2‑second dataset ready:",
      len(os.listdir(TEST_SEG_CLEAN_DIR)), "segments")

Found 237 matched .flac basenames.


Splitting into 2s segments: 100%|██████████| 237/237 [00:38<00:00,  6.23it/s]

✓ RealMix 2‑second dataset ready: 913 segments


In [26]:
#Sample newly created two sec audio
import os

# Get the first audio file path within the directory
clean_audio_file = os.path.join(TEST_SEG_CLEAN_DIR, os.listdir(TEST_SEG_CLEAN_DIR)[6])

print("Clean")
display(Audio(clean_audio_file))  # Display the audio using the file path

noisy_audio_file = os.path.join(TEST_SEG_NOISY_DIR, os.listdir(TEST_SEG_NOISY_DIR)[6])

print("Noisy")
display(Audio(noisy_audio_file))  # Display the audio using the file path
#

Clean


Noisy
